# Methods and results

`nearest` takes the nearest node and time step, `interp` weights several of them.

In [1]:
import numpy as np
import pandas as pd

from floatmatcher.floatmatcher import FloatMatcher

### Points and product

In [2]:
df = pd.DataFrame({
    "longitude": [-38.2, -35.6, -31.9, -27.4, -23.1],
    "latitude":  [31.5, 34.2, 37.8, 40.1, 43.6],
    "date": pd.to_datetime(["2018-01-02 03:10", "2018-01-03 14:45", "2018-01-05 08:20",
                            "2018-01-06 21:05", "2018-01-08 11:30"]),
})

my_fm = FloatMatcher()
my_fm.set_points(df)
my_fm.set_product(type="era5-hourly", source="local", selected_variables=["sst"],
                  path="/runtime/data/era5_daily",  # adapt to your archive
                  pattern="{year}/{month:02d}/era5_single-levels_{year}{month:02d}{day:02d}.nc",
                  )

### Nearest

In [3]:
my_fm.set_method(type="nearest", radius=25, time_gap=np.timedelta64(12, "h"))
my_fm.match()
my_fm.result


    variables: dict_keys(['sst'])                 
    mean_dist:10.72 (km)
    mean_time_delta:9360.0 (seconds)                 
    nb_match_found:5

In [4]:
# one value per point, with its distance to the node and its gap to the time step
result = my_fm.result
pd.DataFrame({
    "sst": result.values["sst"],
    "distance_km": result.distance_km,
    "time_gap_h": result.time_delta / 3600,
    "valid": result.valid,
})

,sst,distance_km,time_gap_h,valid
0,295.034180,4.740463,3.166667,True
1,293.568359,10.744308,2.750000,True
2,291.394775,10.399949,3.666667,True
3,290.468750,14.003337,2.916667,True
4,288.234375,13.732890,0.500000,True


In [5]:
nearest = my_fm.reinject()  # the input DataFrame, with sst_coloc added

### Interpolation

In [7]:
my_fm.set_method(type="interp",
                 radius=50,
                 time_gap=np.timedelta64(12, "h"),
                 k_nearest=4,     # nodes retained for interpolation
                 n_frames=2,      # time steps weightused for interpolation
                 kernel="idw",    # inverse distance weighting
                 power=2,
                 )
my_fm.match()
interp = my_fm.reinject()

pd.DataFrame({"nearest": nearest["sst_coloc"], "interp": interp["sst_coloc"]})

,nearest,interp
0,295.034180,295.032135
1,293.568359,293.522919
2,291.394775,291.390686
3,290.468750,290.411133
4,288.234375,288.258575


### Radius and time gap

Beyond `radius` or `time_gap`, a point is not matched: NaN and `valid=False`.

In [8]:
my_fm.set_method(type="nearest", radius=12, time_gap=np.timedelta64(4, "h"))
my_fm.match()
my_fm.reinject()  # points 3 and 4 are more than 12 km from a node

,longitude,latitude,date,sst_coloc
0,-38.2,31.5,2018-01-02 03:10:00,295.034180
1,-35.6,34.2,2018-01-03 14:45:00,293.568359
2,-31.9,37.8,2018-01-05 08:20:00,291.394775
3,-27.4,40.1,2018-01-06 21:05:00,NaN
4,-23.1,43.6,2018-01-08 11:30:00,NaN


`reinject()` gives back the type of the input: `to_dataframe()` for a DataFrame, `to_dataset()` for a Dataset.